In [ ]:
import torch
import torch.nn.functional as F

from src.temporal_score_cam import TemporalScoreCAM
from src.utils.config_utils import setup_config_dictionary
from src.utils.save_utils import interactive_load_model
from src.utils.plot_utils import plot_frames, plot_cams, plot_combined_frames, combine_polarities
from src.utils.snn_utils import compute_raw_mem, compute_delta_raw_mem

from src.utils.data_utils import get_dataloaders
from src.utils.model_utils import get_model

config = setup_config_dictionary()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(torch.seed())
# torch.manual_seed(4762538657146009718)

print("Run this one cell at a time, otherwise interactive loading will not work.")

In [ ]:
gesture_map = {
    0: "hand_clapping",
    1: "right_hand_wave",
    2: "left_hand_wave",
    3: "right_hand_clockwise",
    4: "right_hand_counter_clockwise",
    5: "left_hand_clockwise",
    6: "left_hand_counter_clockwise",
    7: "forearm_roll_forward",
    8: "forearm_roll_backward",
    9: "drums",
    10: "guitar",
    #11: "random_other_gestures", # -> not implemented here
}

# load the model
train_loader, test_loader = get_dataloaders(
    config["DATASET_NAME"], 
    1, 
    False,
)

event_batches, _ = next(iter(train_loader))
num_channels = event_batches.shape[2]

model = get_model(
        config["MODEL_NAME"],
        num_classes=len(train_loader.dataset.classes),
        num_channels=num_channels,
        time_steps=config["N_TIME_BINS"],
        beta=config["BETA"],
        enable_bntt=config["ENABLE_BNTT"],
    ).to(device)

interactive_load_model(model, config["MODEL_SAVE_DIR"], config["MODEL_NAME"])

In [ ]:
# Visualize using our ScoreCAM implementation
score_cam = TemporalScoreCAM(model)
conv_layers = [module for module in model.modules() if isinstance(module, torch.nn.Conv2d)]
print("Total convolutional layers:", len(conv_layers))

model.eval()
with torch.no_grad():
    for batch_idx, (event_batches, labels) in enumerate(test_loader):
        event_batches = event_batches.to(device)
        labels = labels.to(device)
        if isinstance(model, tuple):
            out_spikes, out_mem = model(event_batches)
            raw_mem = compute_raw_mem(out_spikes, out_mem)
            delta_raw_mem = compute_delta_raw_mem(raw_mem)
            output_class = F.softmax(out_spikes.sum(dim=1), dim=1).cpu().numpy()
        else:
            outputs = model(event_batches)
            delta_outputs = compute_delta_raw_mem(outputs)
            output_class = F.softmax(outputs[:, -1, :], dim=1).cpu().numpy()

        print(f"Output probabilities: {[f'{p:.2%}' for p in output_class[0]]}\n")

        sorted_indices = output_class[0].argsort()[::-1]  # Descending order
        predicted_label = int(sorted_indices[0])
        second_label = int(sorted_indices[1])
        correct_prediction = (predicted_label == labels.item())
        if correct_prediction:
            print(f"Correct prediction with label {predicted_label} -> {gesture_map[predicted_label]}")
        else:
            print(f"Predicted label {predicted_label}, real label: {labels.item()}\nPredicted gesture: {gesture_map[predicted_label]}, real gesture: {gesture_map[labels.item()]}")

        combined_frames = combine_polarities(event_batches[0].cpu().numpy())
        plot_combined_frames(combined_frames)
        step = 2 if len(conv_layers) >= 3 else 1
        print("Per-layer CAMs, from earlier to later convolutional layer:")
        for layer_idx in range(0, len(conv_layers), step):
            target_layer = conv_layers[layer_idx]
            cam = score_cam.generate_cam(event_batches, target_layer=target_layer, last_layer_spikes=False)
            print(f"Layer {layer_idx+1}:")
            print(f"Cams for predicted movement: {gesture_map[predicted_label]}")
            plot_cams(cam, class_idx=predicted_label, frames=combined_frames)
            if correct_prediction:
                print(f"Cams for second most likely movement: {gesture_map[second_label]}")
                plot_cams(cam, class_idx=second_label, frames=combined_frames)
            else: # incorrect prediction
                print(f"Cams for real movement: {gesture_map[labels.item()]}")
                plot_cams(cam, class_idx=labels.item(), frames=combined_frames)
        break
